# 06 - Experiments

## Objetivo

Treinar e avaliar um MLP com embeddings em PyTorch para ranquear produtos candidatos no dataset temporal produzido no notebook `04-feature-engineering.ipynb`.

Este notebook consome candidatos ja gerados. O modelo aprende apenas a ordenar os pares `user_window_id-product_id`; ele nao gera novos candidatos.

## Inputs

- `data/features/temporal_modeling_dataset_v1/`

## Outputs

- Melhor checkpoint local em `models/mlp_temporal_v1/best_model.pt`.
- Configuracao do MLP em `models/mlp_temporal_v1/config.json`.
- Runs no MLflow com metricas, comparacoes e metadados do experimento.

## Regras de avaliacao

- Nao ha split aleatorio por linha.
- Os splits `train`, `validation` e `test` vem prontos no dataset temporal.
- O ranking e feito dentro de cada `user_window_id`.
- `target` e usado apenas para treino e avaliacao, nunca como feature.
- `ndcg@10` e a metrica principal para selecao de modelo.
- O split `test` fica reservado para avaliacao final offline do modelo escolhido.

## Guarda anti-leakage

As colunas de auditoria `split`, `window_number`, `user_window_id`, `target_order_id`, `target_order_number`, `history_start_order_number` e `history_end_order_number` devem ser preservadas para controle e avaliacao, mas nao usadas diretamente como features do modelo.

`candidate_source` e `candidate_rank` carregam sinal forte da estrategia de geracao de candidatos. Elas serao tratadas como uma escolha experimental explicita, comparando versoes com e sem esses sinais.

---

## 1. Setup inicial

In [1]:
import json
import os
import random
import subprocess
import time
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import torch
import torch.nn as nn
from dotenv import load_dotenv
from torch.utils.data import DataLoader, IterableDataset

pd.set_option("display.max_columns", 120)

In [2]:
PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
FEATURES_DIR = DATA_DIR / "features"
MODELS_DIR = PROJECT_ROOT / "models"
MODEL_RUN_DIR = MODELS_DIR / "mlp_temporal_v1"

BEST_MODEL_PATH = MODEL_RUN_DIR / "best_model.pt"
CONFIG_PATH = MODEL_RUN_DIR / "config.json"


TEMPORAL_MODELING_DATASET_DIR = FEATURES_DIR / "temporal_modeling_dataset_v1"
TEMPORAL_MODELING_DATASET_DVC_PATH = FEATURES_DIR / "temporal_modeling_dataset_v1.dvc"

EXPERIMENT_NAME = "mlp-market-recommender-system-temporal-v1"
RUN_MLFLOW = True

RANDOM_SEED = 42
K_VALUES = [5, 10, 20]
PRIMARY_K = 10
PRIMARY_METRIC = "ndcg"

BATCH_SIZE = 8192
MAX_EPOCHS = 30

PATIENCE = 10
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
EMBEDDING_DIM = 32
HIDDEN_DIMS = [128, 64]
DROPOUT = 0.25

LR_SCHEDULER_PATIENCE = 5
LR_SCHEDULER_FACTOR = 0.5

MODEL_RUN_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


seed_everything(RANDOM_SEED)
device = get_device()

torch_generator = torch.Generator()
torch_generator.manual_seed(RANDOM_SEED)

experiment_config = {
    "random_seed": RANDOM_SEED,
    "k_values": K_VALUES,
    "primary_k": PRIMARY_K,
    "primary_metric": PRIMARY_METRIC,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "lr_scheduler_patience": LR_SCHEDULER_PATIENCE,
    "lr_scheduler_factor": LR_SCHEDULER_FACTOR,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "embedding_dim": EMBEDDING_DIM,
    "hidden_dims": HIDDEN_DIMS,
    "dropout": DROPOUT,
    "device": str(device),
}

experiment_config

{'random_seed': 42,
 'k_values': [5, 10, 20],
 'primary_k': 10,
 'primary_metric': 'ndcg',
 'batch_size': 8192,
 'max_epochs': 30,
 'patience': 10,
 'lr_scheduler_patience': 5,
 'lr_scheduler_factor': 0.5,
 'learning_rate': 0.001,
 'weight_decay': 1e-05,
 'embedding_dim': 32,
 'hidden_dims': [128, 64],
 'dropout': 0.25,
 'device': 'cuda'}

---

## 2. Carregamento do dataset de modelagem temporal

In [4]:
part_paths = sorted(TEMPORAL_MODELING_DATASET_DIR.glob("*.parquet"))
modeling_dataset = ds.dataset(TEMPORAL_MODELING_DATASET_DIR, format="parquet")
dataset_columns = modeling_dataset.schema.names

print(f"Particoes: {len(part_paths):,}")
print(f"Colunas: {len(dataset_columns):,}")

Particoes: 47
Colunas: 32


---

## 3. Definicao das features

A primeira versao do MLP usa embeddings para identificadores principais e features numericas historicas calculadas por janela temporal.

In [5]:
EMBEDDING_COLUMNS = [
    "user_id",
    "product_id",
    "aisle_id",
    "department_id",
]

NUMERIC_FEATURE_COLUMNS = [
    "history_order_count",
    "history_unique_products",
    "user_prior_order_count",
    "user_avg_basket_size",
    "user_avg_days_between_orders",
    "user_reorder_rate",
    "user_total_items",
    "user_has_single_prior_order",
    "user_product_purchase_count",
    "user_product_reorder_count",
    "user_product_avg_add_to_cart_order",
    "user_product_orders_since_last_purchase",
    "user_product_days_since_last_purchase",
    "user_product_was_bought_before",
    "user_product_purchase_share",
    "user_aisle_purchase_count",
    "user_department_purchase_count",
]

CANDIDATE_SIGNAL_COLUMNS = [
    "candidate_rank",
    "candidate_source",
]

EVALUATION_COLUMNS = [
    "split",
    "user_window_id",
    "target_order_id",
    "product_id",
    "target",
]

TARGET_COLUMN = "target"

feature_config = {
    "embedding_columns": EMBEDDING_COLUMNS,
    "numeric_feature_columns": NUMERIC_FEATURE_COLUMNS,
    "candidate_signal_columns": CANDIDATE_SIGNAL_COLUMNS,
    "evaluation_columns": EVALUATION_COLUMNS,
    "target_column": TARGET_COLUMN,
}

---

## 4. Preprocessamento baseado no treino

Os mapeamentos categoricos e as estatisticas de normalizacao sao ajustados usando apenas `split = train`.

IDs desconhecidos em `validation` e `test` recebem indice `0`, reservado para `<UNK>`.

### 4.1 Colunas usadas pelo modelo

Esta célula consolida as colunas necessárias para carregar os dados de treino, validação e teste.

`MODEL_COLUMNS` combina os identificadores usados em embeddings, as features numéricas e as colunas preservadas para avaliação. O índice `0` fica reservado para `<UNK>`, usado quando algum ID aparecer em validação/teste sem ter sido visto no treino.

In [6]:
MODEL_COLUMNS = list(
    dict.fromkeys(EMBEDDING_COLUMNS + NUMERIC_FEATURE_COLUMNS + EVALUATION_COLUMNS)
)

UNK_INDEX = 0

### 4.2 Mapeamentos para embeddings

As camadas de embedding precisam receber índices inteiros compactos, não os IDs originais do Instacart.

Os mapeamentos são ajustados somente com `split = train`. Assim, IDs desconhecidos em `validation` e `test` não influenciam o vocabulário do modelo e serão mapeados para `<UNK>`.

In [7]:
def build_category_maps(part_paths, categorical_columns):
    category_values = {column: set() for column in categorical_columns}

    for part_path in part_paths:
        part_df = pd.read_parquet(
            part_path,
            columns=["split", *categorical_columns],
        )
        train_df = part_df[part_df["split"] == "train"]

        for column in categorical_columns:
            category_values[column].update(
                train_df[column].dropna().astype(int).unique().tolist()
            )

    category_maps = {}

    for column, values in category_values.items():
        sorted_values = sorted(values)
        category_maps[column] = {
            value: idx for idx, value in enumerate(sorted_values, start=1)
        }

    return category_maps


category_maps = build_category_maps(
    part_paths=part_paths,
    categorical_columns=EMBEDDING_COLUMNS,
)

embedding_cardinalities = {
    column: len(mapping) + 1 for column, mapping in category_maps.items()
}

embedding_cardinalities

{'user_id': 115910, 'product_id': 49047, 'aisle_id': 135, 'department_id': 22}

### 4.3 Estatísticas das features numéricas

As features numéricas têm escalas diferentes, então serão padronizadas antes de entrar no MLP.

A média e o desvio padrão são calculados apenas no `split = train`. Os mesmos valores serão reutilizados em validação e teste para evitar vazamento de informação.

In [8]:
def compute_numeric_scaler_stats(part_paths, numeric_columns):
    sums = pd.Series(0.0, index=numeric_columns)
    squared_sums = pd.Series(0.0, index=numeric_columns)
    count = 0

    for part_path in part_paths:
        part_df = pd.read_parquet(
            part_path,
            columns=["split", *numeric_columns],
        )
        train_df = part_df[part_df["split"] == "train"][numeric_columns].fillna(0)

        sums += train_df.sum()
        squared_sums += (train_df**2).sum()
        count += len(train_df)

    means = sums / count
    variances = (squared_sums / count) - (means**2)
    stds = np.sqrt(variances.clip(lower=0)).replace(0, 1)

    return {
        "mean": means.to_dict(),
        "std": stds.to_dict(),
    }


scaler_stats = compute_numeric_scaler_stats(
    part_paths=part_paths,
    numeric_columns=NUMERIC_FEATURE_COLUMNS,
)

pd.DataFrame(scaler_stats)

,mean,std
history_order_count,14.767658,17.051211
history_unique_products,59.773188,57.117554
user_prior_order_count,14.767658,17.051211
user_avg_basket_size,9.929746,5.986013
user_avg_days_between_orders,10.754598,5.600959
user_reorder_rate,0.400032,0.239708
user_total_items,148.391958,204.172407
user_has_single_prior_order,0.053658,0.225342
user_product_purchase_count,0.722353,2.295877
user_product_reorder_count,0.442609,2.105227


### 4.4 Funções de transformação

Estas funções aplicam os preprocessamentos definidos nas células anteriores.

`map_categorical_columns` converte IDs reais para índices de embedding e envia IDs desconhecidos para `<UNK>`. `scale_numeric_columns` aplica a padronização das features numéricas usando as estatísticas calculadas no treino.

In [9]:
def map_categorical_columns(df, category_maps, categorical_columns):
    mapped_df = pd.DataFrame(index=df.index)

    for column in categorical_columns:
        mapped_df[column] = (
            df[column].map(category_maps[column]).fillna(UNK_INDEX).astype("int64")
        )

    return mapped_df


def scale_numeric_columns(df, numeric_columns, scaler_stats):
    numeric_df = df[numeric_columns].fillna(0).astype("float32")

    means = pd.Series(scaler_stats["mean"])
    stds = pd.Series(scaler_stats["std"])

    return ((numeric_df - means) / stds).astype("float32")

---

## 5. Dataset e DataLoader

O dataset temporal e grande, entao o treino sera feito por leitura incremental das particoes Parquet.

O `IterableDataset` abaixo le uma particao por vez, filtra o split desejado, aplica os mapeamentos de embeddings e a normalizacao numerica, e entrega batches prontos para o PyTorch.

In [10]:
class TemporalParquetDataset(IterableDataset):
    def __init__(
        self,
        part_paths,
        split,
        model_columns,
        embedding_columns,
        numeric_columns,
        category_maps,
        scaler_stats,
        batch_size,
        shuffle_partitions=False,
        seed=42,
        include_metadata=False,
    ):
        self.part_paths = list(part_paths)
        self.split = split
        self.model_columns = model_columns
        self.embedding_columns = embedding_columns
        self.numeric_columns = numeric_columns
        self.category_maps = category_maps
        self.scaler_stats = scaler_stats
        self.batch_size = batch_size
        self.shuffle_partitions = shuffle_partitions
        self.seed = seed
        self.include_metadata = include_metadata

    def __iter__(self):
        part_paths = self.part_paths.copy()

        if self.shuffle_partitions:
            rng = np.random.default_rng(self.seed)
            rng.shuffle(part_paths)

        for part_path in part_paths:
            part_df = pd.read_parquet(part_path, columns=self.model_columns)
            part_df = part_df[part_df["split"] == self.split].copy()

            if part_df.empty:
                continue

            if self.shuffle_partitions:
                part_df = part_df.sample(
                    frac=1,
                    random_state=self.seed,
                ).reset_index(drop=True)

            categorical_df = map_categorical_columns(
                df=part_df,
                category_maps=self.category_maps,
                categorical_columns=self.embedding_columns,
            )
            numeric_df = scale_numeric_columns(
                df=part_df,
                numeric_columns=self.numeric_columns,
                scaler_stats=self.scaler_stats,
            )
            target = part_df[TARGET_COLUMN].astype("float32").to_numpy()

            for start_idx in range(0, len(part_df), self.batch_size):
                end_idx = start_idx + self.batch_size

                batch = {
                    "categorical": torch.tensor(
                        categorical_df.iloc[start_idx:end_idx].to_numpy(),
                        dtype=torch.long,
                    ),
                    "numeric": torch.tensor(
                        numeric_df.iloc[start_idx:end_idx].to_numpy(),
                        dtype=torch.float32,
                    ),
                    "target": torch.tensor(
                        target[start_idx:end_idx],
                        dtype=torch.float32,
                    ),
                }

                if self.include_metadata:
                    batch["metadata"] = part_df.iloc[start_idx:end_idx][
                        EVALUATION_COLUMNS
                    ].copy()

                yield batch

In [11]:
train_dataset = TemporalParquetDataset(
    part_paths=part_paths,
    split="train",
    model_columns=MODEL_COLUMNS,
    embedding_columns=EMBEDDING_COLUMNS,
    numeric_columns=NUMERIC_FEATURE_COLUMNS,
    category_maps=category_maps,
    scaler_stats=scaler_stats,
    batch_size=BATCH_SIZE,
    shuffle_partitions=True,
    seed=RANDOM_SEED,
    include_metadata=False,
)

validation_dataset = TemporalParquetDataset(
    part_paths=part_paths,
    split="validation",
    model_columns=MODEL_COLUMNS,
    embedding_columns=EMBEDDING_COLUMNS,
    numeric_columns=NUMERIC_FEATURE_COLUMNS,
    category_maps=category_maps,
    scaler_stats=scaler_stats,
    batch_size=BATCH_SIZE,
    shuffle_partitions=False,
    seed=RANDOM_SEED,
    include_metadata=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=None,
    num_workers=0,
    generator=torch_generator,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=None,
    num_workers=0,
    generator=torch_generator,
)

In [12]:
sample_batch = next(iter(train_loader))

{
    "categorical_shape": tuple(sample_batch["categorical"].shape),
    "numeric_shape": tuple(sample_batch["numeric"].shape),
    "target_shape": tuple(sample_batch["target"].shape),
}

{'categorical_shape': (8192, 4),
 'numeric_shape': (8192, 17),
 'target_shape': (8192,)}

---

## 6. Arquitetura MLP

A primeira arquitetura neural sera simples: embeddings para as colunas categoricas, concatenacao com features numericas normalizadas e um MLP pequeno para gerar um logit por candidato.

A saida nao passa por sigmoid, porque o treino usara `BCEWithLogitsLoss`.

In [13]:
class MLPRecommender(nn.Module):
    def __init__(
        self,
        embedding_cardinalities,
        embedding_columns,
        embedding_dim,
        numeric_input_dim,
        hidden_dims,
        dropout,
    ):
        super().__init__()

        self.embedding_columns = embedding_columns
        self.embeddings = nn.ModuleList(
            [
                nn.Embedding(
                    num_embeddings=embedding_cardinalities[column],
                    embedding_dim=embedding_dim,
                    padding_idx=UNK_INDEX,
                )
                for column in embedding_columns
            ]
        )

        input_dim = (len(embedding_columns) * embedding_dim) + numeric_input_dim

        layers = []
        previous_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.extend(
                [
                    nn.Linear(previous_dim, hidden_dim),
                    nn.ReLU(),
                    nn.Dropout(dropout),
                ]
            )
            previous_dim = hidden_dim

        layers.append(nn.Linear(previous_dim, 1))

        self.mlp = nn.Sequential(*layers)

    def forward(self, categorical_inputs, numeric_inputs):
        embedded_inputs = [
            embedding(categorical_inputs[:, idx])
            for idx, embedding in enumerate(self.embeddings)
        ]

        x = torch.cat([*embedded_inputs, numeric_inputs], dim=1)
        logits = self.mlp(x).squeeze(1)

        return logits

In [14]:
model = MLPRecommender(
    embedding_cardinalities=embedding_cardinalities,
    embedding_columns=EMBEDDING_COLUMNS,
    embedding_dim=EMBEDDING_DIM,
    numeric_input_dim=len(NUMERIC_FEATURE_COLUMNS),
    hidden_dims=HIDDEN_DIMS,
    dropout=DROPOUT,
).to(device)

trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)

trainable_parameters

5310657

In [15]:
sample_batch = next(iter(train_loader))

model.eval()

with torch.no_grad():
    sample_logits = model(
        categorical_inputs=sample_batch["categorical"].to(device),
        numeric_inputs=sample_batch["numeric"].to(device),
    )

{
    "logits_shape": tuple(sample_logits.shape),
    "logits_min": float(sample_logits.min().cpu()),
    "logits_max": float(sample_logits.max().cpu()),
}

{'logits_shape': (8192,),
 'logits_min': -0.375997394323349,
 'logits_max': 0.16357286274433136}

---

## 7. Loss, desbalanceamento e otimizador

O modelo sera treinado como um classificador binario pointwise: para cada par `user_window_id-product_id`, ele aprende um score maior quando o produto foi comprado no pedido alvo e menor caso contrario.

A loss escolhida e `BCEWithLogitsLoss`. Ela recebe diretamente os logits produzidos pela ultima camada linear do MLP e aplica internamente a sigmoid antes de calcular a entropia cruzada binaria. Essa abordagem e numericamente mais estavel do que aplicar `sigmoid` manualmente no modelo e depois usar `BCELoss`.

O dataset e desbalanceado: cada janela possui muitos candidatos negativos e poucos positivos. Por isso, usamos `pos_weight` calculado apenas no `split = train`, com a razao `negativos / positivos`. A vantagem e reduzir a tendencia do modelo a favorecer sempre a classe negativa. A desvantagem e que os scores podem ficar menos calibrados como probabilidade. Neste projeto isso e aceitavel, porque o objetivo principal e ordenar candidatos dentro de cada `user_window_id`, nao produzir probabilidades perfeitamente calibradas.

O otimizador escolhido e `AdamW`. Ele e adequado para MLPs com embeddings porque adapta o passo de aprendizado por parametro, o que ajuda quando o modelo combina embeddings de alta cardinalidade com features numericas. Em relacao ao `Adam`, o `AdamW` aplica `weight_decay` de forma desacoplada, o que tende a ser uma regularizacao mais correta. Em relacao ao `SGD`, costuma exigir menos tuning manual e convergir mais rapido neste tipo de modelo. A desvantagem e adicionar um pouco mais de memoria/estado interno por parametro, mas o trade-off e adequado para esta primeira versao.

In [16]:
def compute_pos_weight(part_paths):
    positives = 0
    rows = 0

    for part_path in part_paths:
        part_df = pd.read_parquet(part_path, columns=["split", "target"])
        train_target = part_df.loc[part_df["split"] == "train", "target"]

        positives += int(train_target.sum())
        rows += len(train_target)

    negatives = rows - positives
    pos_weight_value = negatives / positives

    return torch.tensor([pos_weight_value], dtype=torch.float32, device=device)


pos_weight = compute_pos_weight(part_paths)

float(pos_weight.cpu())

25.63154411315918

In [17]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    patience=LR_SCHEDULER_PATIENCE,
    factor=LR_SCHEDULER_FACTOR,
)

{
    "loss": criterion.__class__.__name__,
    "optimizer": optimizer.__class__.__name__,
    "scheduler": scheduler.__class__.__name__,
    "learning_rate": optimizer.param_groups[0]["lr"],
    "weight_decay": optimizer.param_groups[0]["weight_decay"],
    "pos_weight": float(pos_weight.cpu()),
}

{'loss': 'BCEWithLogitsLoss',
 'optimizer': 'AdamW',
 'scheduler': 'ReduceLROnPlateau',
 'learning_rate': 0.001,
 'weight_decay': 1e-05,
 'pos_weight': 25.63154411315918}

---

## 8. Metricas de ranking

A avaliacao do MLP e feita por ranking dentro de cada `user_window_id`.

Nesta etapa usamos apenas metricas locais, isto e, metricas calculadas sobre os positivos que chegaram ao dataset de candidatos. O `recall_global` fica fora do fluxo principal porque mede o sistema completo `candidate generation + ranking`, ja analisado no notebook de baseline.

### 8.1 Métricas de uma janela

In [18]:
def dcg_at_k(relevance, k):
    relevance = np.asarray(relevance[:k], dtype=float)

    if relevance.size == 0:
        return 0.0

    discounts = 1.0 / np.log2(np.arange(2, relevance.size + 2))
    return float(np.sum(relevance * discounts))


def compute_window_metrics(window_df, k):
    ranked_df = window_df.sort_values("score", ascending=False)
    relevance = ranked_df["target"].to_numpy()

    positives = int(window_df["target"].sum())

    if positives == 0:
        return None

    top_k_relevance = relevance[:k]
    hits = int(top_k_relevance.sum())

    ideal_relevance = np.ones(min(positives, k))
    ideal_dcg = dcg_at_k(ideal_relevance, k)

    return {
        "ndcg": dcg_at_k(relevance, k) / ideal_dcg if ideal_dcg > 0 else 0.0,
        "hit_rate": float(hits > 0),
        "recall_local": hits / positives,
        "precision": hits / k,
    }

### 8.2 Métricas por split

In [19]:
def evaluate_ranking_predictions(predictions_df, k_values):
    results = []

    for split_name, split_df in predictions_df.groupby("split"):
        for k in k_values:
            window_metrics = []

            for _, window_df in split_df.groupby("user_window_id", sort=False):
                metrics = compute_window_metrics(window_df, k)

                if metrics is not None:
                    window_metrics.append(metrics)

            metrics_df = pd.DataFrame(window_metrics)

            results.append(
                {
                    "split": split_name,
                    "k": k,
                    "windows_evaluated": len(metrics_df),
                    "ndcg": metrics_df["ndcg"].mean(),
                    "hit_rate": metrics_df["hit_rate"].mean(),
                    "recall_local": metrics_df["recall_local"].mean(),
                    "precision": metrics_df["precision"].mean(),
                }
            )

    return pd.DataFrame(results)

### 8.3 Função de predição

In [20]:
def predict_loader(model, loader, device):
    model.eval()
    prediction_parts = []

    with torch.no_grad():
        for batch in loader:
            logits = model(
                categorical_inputs=batch["categorical"].to(device),
                numeric_inputs=batch["numeric"].to(device),
            )

            metadata_df = batch["metadata"].copy()
            metadata_df["score"] = logits.cpu().numpy()

            prediction_parts.append(metadata_df)

    return pd.concat(prediction_parts, ignore_index=True)

### 8.4 Validação das funções

In [21]:
validation_predictions_df = predict_loader(
    model=model,
    loader=validation_loader,
    device=device,
)

validation_metrics_df = evaluate_ranking_predictions(
    predictions_df=validation_predictions_df,
    k_values=K_VALUES,
)

validation_metrics_df

,split,k,windows_evaluated,ndcg,hit_rate,recall_local,precision
0,validation,5,112240,0.070468,0.251042,0.040675,0.064052
1,validation,10,112240,0.075876,0.380952,0.072780,0.058387
2,validation,20,112240,0.096150,0.539932,0.131088,0.052859


---

## 9. Treino com scheduler e early stopping

O treino monitora `ndcg@10` no split de validacao.

O learning rate e controlado por `ReduceLROnPlateau`: quando a metrica de validacao fica sem melhora por `LR_SCHEDULER_PATIENCE` epocas, o scheduler reduz o learning rate por `LR_SCHEDULER_FACTOR`.

O early stopping usa um contador unico de epocas sem melhora. Esse contador zera apenas quando o modelo melhora a melhor metrica de validacao observada. A reducao do learning rate nao zera o contador; ela apenas da ao modelo uma chance de melhorar antes que `PATIENCE` seja atingido.

O melhor checkpoint e salvo localmente em `models/mlp_temporal_v1/best_model.pt`.

### 9.1 Funções de treino

In [22]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    total_examples = 0

    for batch in loader:
        categorical_inputs = batch["categorical"].to(device)
        numeric_inputs = batch["numeric"].to(device)
        targets = batch["target"].to(device)

        optimizer.zero_grad()

        logits = model(
            categorical_inputs=categorical_inputs,
            numeric_inputs=numeric_inputs,
        )
        loss = criterion(logits, targets)

        loss.backward()
        optimizer.step()

        batch_size = len(targets)
        total_loss += float(loss.detach().cpu()) * batch_size
        total_examples += batch_size

    return total_loss / total_examples

In [23]:
def save_model_checkpoint(path, model, epoch, best_metric, experiment_config):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "epoch": epoch,
        "best_metric": best_metric,
        "experiment_config": experiment_config,
        "feature_config": feature_config,
        "embedding_cardinalities": embedding_cardinalities,
    }

    torch.save(checkpoint, path)

### 9.2 Loop de treino

In [24]:
best_validation_metric = -np.inf
best_epoch = 0
epochs_without_improvement = 0
last_lr = optimizer.param_groups[0]["lr"]

training_history = []

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.time()

    print(f"\nEpoch {epoch}/{MAX_EPOCHS} | lr={last_lr:.6f}")

    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
    )

    validation_predictions_df = predict_loader(
        model=model,
        loader=validation_loader,
        device=device,
    )
    validation_metrics_df = evaluate_ranking_predictions(
        predictions_df=validation_predictions_df,
        k_values=K_VALUES,
    )

    validation_metric = float(
        validation_metrics_df.loc[
            validation_metrics_df["k"] == PRIMARY_K,
            PRIMARY_METRIC,
        ].iloc[0]
    )

    improved = validation_metric > best_validation_metric

    if improved:
        best_validation_metric = validation_metric
        best_epoch = epoch
        epochs_without_improvement = 0

        save_model_checkpoint(
            path=BEST_MODEL_PATH,
            model=model,
            epoch=epoch,
            best_metric=best_validation_metric,
            experiment_config=experiment_config,
        )

        print(
            f"  new best {PRIMARY_METRIC}@{PRIMARY_K}: "
            f"{best_validation_metric:.6f} | checkpoint saved"
        )
    else:
        epochs_without_improvement += 1

    scheduler.step(validation_metric)
    current_lr = optimizer.param_groups[0]["lr"]

    if current_lr < last_lr:
        print(f"  learning rate reduced: {last_lr:.6f} -> {current_lr:.6f}")
        last_lr = current_lr

    epoch_seconds = time.time() - epoch_start

    training_history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            f"validation_{PRIMARY_METRIC}_at_{PRIMARY_K}": validation_metric,
            "best_epoch": best_epoch,
            "best_validation_metric": best_validation_metric,
            "learning_rate": current_lr,
            "epochs_without_improvement": epochs_without_improvement,
            "epoch_seconds": epoch_seconds,
        }
    )

    print(
        f"  train_loss={train_loss:.5f} | "
        f"validation_{PRIMARY_METRIC}@{PRIMARY_K}={validation_metric:.6f} | "
        f"best={best_validation_metric:.6f} epoch={best_epoch} | "
        f"sem_melhora={epochs_without_improvement}/{PATIENCE} | "
        f"time={epoch_seconds:.1f}s"
    )

    if epochs_without_improvement >= PATIENCE:
        print("  early stopping triggered")
        break

training_history_df = pd.DataFrame(training_history)
training_history_df.tail()


Epoch 1/30 | lr=0.001000


  new best ndcg@10: 0.499057 | checkpoint saved
  train_loss=0.87518 | validation_ndcg@10=0.499057 | best=0.499057 epoch=1 | sem_melhora=0/10 | time=124.7s

Epoch 2/30 | lr=0.001000


  new best ndcg@10: 0.499856 | checkpoint saved
  train_loss=0.85140 | validation_ndcg@10=0.499856 | best=0.499856 epoch=2 | sem_melhora=0/10 | time=124.2s

Epoch 3/30 | lr=0.001000


  train_loss=0.83179 | validation_ndcg@10=0.499738 | best=0.499856 epoch=2 | sem_melhora=1/10 | time=126.5s

Epoch 4/30 | lr=0.001000


  train_loss=0.81815 | validation_ndcg@10=0.499533 | best=0.499856 epoch=2 | sem_melhora=2/10 | time=126.2s

Epoch 5/30 | lr=0.001000


...

  train_loss=0.78122 | validation_ndcg@10=0.499097 | best=0.499856 epoch=2 | sem_melhora=7/10 | time=126.2s

Epoch 10/30 | lr=0.000500


  train_loss=0.77684 | validation_ndcg@10=0.498747 | best=0.499856 epoch=2 | sem_melhora=8/10 | time=123.2s

Epoch 11/30 | lr=0.000500


  train_loss=0.77298 | validation_ndcg@10=0.498656 | best=0.499856 epoch=2 | sem_melhora=9/10 | time=114.4s

Epoch 12/30 | lr=0.000500


  train_loss=0.76919 | validation_ndcg@10=0.498468 | best=0.499856 epoch=2 | sem_melhora=10/10 | time=113.2s
  early stopping triggered


,epoch,train_loss,validation_ndcg_at_10,best_epoch,best_validation_metric,learning_rate,epochs_without_improvement,epoch_seconds
7,8,0.787613,0.499198,2,0.499856,0.0005,6,129.880913
8,9,0.781224,0.499097,2,0.499856,0.0005,7,126.185267
9,10,0.776840,0.498747,2,0.499856,0.0005,8,123.240964
10,11,0.772979,0.498656,2,0.499856,0.0005,9,114.378994
11,12,0.769194,0.498468,2,0.499856,0.0005,10,113.209711


In [25]:
with CONFIG_PATH.open("w") as file:
    json.dump(experiment_config, file, indent=2)

{
    "best_model_path": str(BEST_MODEL_PATH),
    "config_path": str(CONFIG_PATH),
    "best_epoch": best_epoch,
    f"best_validation_{PRIMARY_METRIC}_at_{PRIMARY_K}": best_validation_metric,
}

{'best_model_path': 'C:\\Users\\erick\\projetos\\mlp-market-recommender-system\\models\\mlp_temporal_v1\\best_model.pt',
 'config_path': 'C:\\Users\\erick\\projetos\\mlp-market-recommender-system\\models\\mlp_temporal_v1\\config.json',
 'best_epoch': 2,
 'best_validation_ndcg_at_10': 0.49985565188976006}

---

## 10. Avaliacao do melhor modelo

In [26]:
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)

best_model = MLPRecommender(
    embedding_cardinalities=embedding_cardinalities,
    embedding_columns=EMBEDDING_COLUMNS,
    embedding_dim=EMBEDDING_DIM,
    numeric_input_dim=len(NUMERIC_FEATURE_COLUMNS),
    hidden_dims=HIDDEN_DIMS,
    dropout=DROPOUT,
).to(device)

best_model.load_state_dict(checkpoint["model_state_dict"])

{
    "checkpoint_epoch": checkpoint["epoch"],
    "checkpoint_best_metric": checkpoint["best_metric"],
}

{'checkpoint_epoch': 2, 'checkpoint_best_metric': 0.49985565188976006}

In [27]:
best_validation_predictions_df = predict_loader(
    model=best_model,
    loader=validation_loader,
    device=device,
)

best_validation_metrics_df = evaluate_ranking_predictions(
    predictions_df=best_validation_predictions_df,
    k_values=K_VALUES,
)

best_validation_metrics_df

,split,k,windows_evaluated,ndcg,hit_rate,recall_local,precision
0,validation,5,112240,0.490125,0.840832,0.340563,0.398737
1,validation,10,112240,0.499856,0.908624,0.487597,0.311855
2,validation,20,112240,0.546364,0.949849,0.644133,0.223228


---

## 11. Comparacao em validacao

A selecao do modelo usa `ndcg@10` no split de validacao.

Nesta etapa comparamos o melhor checkpoint do MLP contra o baseline principal do notebook 05, `ordem_gerador_candidatos`.

In [28]:
# Le o melhor baseline do artefato salvo pelo notebook 05 (fonte unica de
# verdade, em vez de valores copiados manualmente).
baseline_results_df = pd.read_parquet(
    PROJECT_ROOT / "data" / "processed" / "baseline_results.parquet"
)
best_baseline_row = (
    baseline_results_df[
        (baseline_results_df["split"] == "validation")
        & (baseline_results_df["k"] == PRIMARY_K)
    ]
    .sort_values("ndcg", ascending=False)
    .iloc[0]
)
BASELINE_VALIDATION_METRICS = {
    "baseline": str(best_baseline_row["baseline"]),
    "split": "validation",
    "k": PRIMARY_K,
    "ndcg": float(best_baseline_row["ndcg"]),
    "hit_rate": float(best_baseline_row["hit_rate"]),
    "recall_local": float(best_baseline_row["recall_local"]),
    "precision": float(best_baseline_row["precision"]),
}

mlp_validation_at_10 = (
    best_validation_metrics_df[
        (best_validation_metrics_df["split"] == "validation")
        & (best_validation_metrics_df["k"] == PRIMARY_K)
    ]
    .iloc[0]
    .to_dict()
)

validation_comparison_df = pd.DataFrame(
    [
        BASELINE_VALIDATION_METRICS,
        {
            "baseline": "mlp_temporal_v1",
            "split": "validation",
            "k": PRIMARY_K,
            "ndcg": mlp_validation_at_10["ndcg"],
            "hit_rate": mlp_validation_at_10["hit_rate"],
            "recall_local": mlp_validation_at_10["recall_local"],
            "precision": mlp_validation_at_10["precision"],
        },
    ]
)

validation_comparison_df["ndcg_delta_vs_baseline"] = (
    validation_comparison_df["ndcg"] - BASELINE_VALIDATION_METRICS["ndcg"]
)

validation_comparison_df

,baseline,split,k,ndcg,hit_rate,recall_local,precision,ndcg_delta_vs_baseline
0,ordem_gerador_candidatos,validation,10,0.467773,0.864618,0.456651,0.280981,0.000000
1,mlp_temporal_v1,validation,10,0.499856,0.908624,0.487597,0.311855,0.032083


### Leitura da validacao

O MLP temporal superou o baseline principal `ordem_gerador_candidatos` em `ndcg@10`, com ganho absoluto de `+0.032041`.

O ganho tambem aparece em `hit_rate@10`, `recall_local@10` e `precision@10`, indicando que o modelo nao apenas melhora a ordenacao media, mas tambem coloca mais produtos relevantes no top 10.

O melhor checkpoint ocorreu na epoca 3. Depois disso, a loss de treino continuou caindo, mas o `ndcg@10` de validacao piorou, sugerindo overfitting. O early stopping preservou corretamente o melhor modelo observado em validacao.

---

## 12. Avaliacao final no test

O split `test` e usado apenas apos escolher o modelo com base na validacao.

Nesta etapa avaliamos o melhor checkpoint salvo, sem alterar hiperparametros ou selecionar novo modelo.

In [29]:
test_dataset = TemporalParquetDataset(
    part_paths=part_paths,
    split="test",
    model_columns=MODEL_COLUMNS,
    embedding_columns=EMBEDDING_COLUMNS,
    numeric_columns=NUMERIC_FEATURE_COLUMNS,
    category_maps=category_maps,
    scaler_stats=scaler_stats,
    batch_size=BATCH_SIZE,
    shuffle_partitions=False,
    seed=RANDOM_SEED,
    include_metadata=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=None,
    num_workers=0,
    generator=torch_generator,
)

In [30]:
test_predictions_df = predict_loader(
    model=best_model,
    loader=test_loader,
    device=device,
)

test_metrics_df = evaluate_ranking_predictions(
    predictions_df=test_predictions_df,
    k_values=K_VALUES,
)

test_metrics_df

,split,k,windows_evaluated,ndcg,hit_rate,recall_local,precision
0,test,5,112398,0.486036,0.837746,0.332556,0.398539
1,test,10,112398,0.494804,0.905817,0.479327,0.312966
2,test,20,112398,0.540896,0.948407,0.637341,0.225180


---

## 13. Consolidacao das metricas do MLP

Esta tabela consolida as metricas locais do melhor checkpoint em validacao e teste.

A leitura principal continua sendo `ndcg@10`, mas as demais metricas ajudam a entender se o ganho vem de melhor ordenacao, maior cobertura local no top K ou maior precisao.

In [31]:
mlp_metrics_df = pd.concat(
    [
        best_validation_metrics_df,
        test_metrics_df,
    ],
    ignore_index=True,
)

mlp_metrics_df["model"] = "mlp_temporal_v1"

mlp_metrics_df = mlp_metrics_df[
    [
        "model",
        "split",
        "k",
        "windows_evaluated",
        "ndcg",
        "hit_rate",
        "recall_local",
        "precision",
    ]
]

mlp_metrics_df

,model,split,k,windows_evaluated,ndcg,hit_rate,recall_local,precision
0,mlp_temporal_v1,validation,5,112240,0.490125,0.840832,0.340563,0.398737
1,mlp_temporal_v1,validation,10,112240,0.499856,0.908624,0.487597,0.311855
2,mlp_temporal_v1,validation,20,112240,0.546364,0.949849,0.644133,0.223228
3,mlp_temporal_v1,test,5,112398,0.486036,0.837746,0.332556,0.398539
4,mlp_temporal_v1,test,10,112398,0.494804,0.905817,0.479327,0.312966
5,mlp_temporal_v1,test,20,112398,0.540896,0.948407,0.637341,0.225180


### Leitura dos resultados

O MLP temporal superou o baseline principal em validacao, com ganho de `+0.032041` em `ndcg@10`.

No teste, o modelo manteve desempenho proximo ao observado em validacao: `ndcg@10 = 0.494838`, `hit_rate@10 = 0.905973`, `recall_local@10 = 0.479348` e `precision@10 = 0.313011`.

A diferenca entre validacao e teste e pequena, sugerindo que o checkpoint escolhido por validacao generalizou bem para o holdout final. O resultado tambem confirma que a estrategia de recompra e a ordem dos candidatos sao baselines fortes, mas o MLP conseguiu capturar sinal adicional a partir dos embeddings e das features temporais.

---

## 14. Registro no MLflow

Esta etapa registra a run do MLP no MLflow.

O dataset completo nao e enviado como artefato. Os dados sao versionados pelo DVC; o MLflow registra parametros, metricas, Git commit, metadados DVC e pequenos artefatos de configuracao/resultados.

In [32]:
def run_command(command):
    result = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        check=False,
    )

    if result.returncode != 0:
        return None

    return result.stdout.strip()


def get_git_commit():
    return run_command(["git", "rev-parse", "HEAD"])


def read_dvc_metadata(dvc_path):
    if not dvc_path.exists():
        return {}

    return {
        "dvc_path": str(dvc_path.relative_to(PROJECT_ROOT)),
        "dvc_file_content": dvc_path.read_text(),
    }


def configure_mlflow():
    load_dotenv()

    tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
    assert tracking_uri, "MLFLOW_TRACKING_URI nao configurado."

    mlflow.set_tracking_uri(tracking_uri)
    mlflow.set_experiment(EXPERIMENT_NAME)

    return tracking_uri

In [33]:
git_commit = get_git_commit()
dvc_metadata = read_dvc_metadata(TEMPORAL_MODELING_DATASET_DVC_PATH)

mlflow_payloads = {
    "feature_config": feature_config,
    "embedding_cardinalities": embedding_cardinalities,
    "scaler_stats": scaler_stats,
    "dvc_metadata": dvc_metadata,
}

git_commit

'0527fe9e21a90b15a5424473b32d961be5e6aa7f'

In [34]:
def log_mlp_run():
    tracking_uri = configure_mlflow()

    with mlflow.start_run(run_name="mlp_temporal_v1_catfix"):
        mlflow.log_params(experiment_config)

        mlflow.log_param("model_name", "mlp_temporal_v1")
        mlflow.log_param(
            "dataset_path", str(TEMPORAL_MODELING_DATASET_DIR.relative_to(PROJECT_ROOT))
        )
        mlflow.log_param("best_epoch", int(best_epoch))
        mlflow.log_param(
            "best_model_path", str(BEST_MODEL_PATH.relative_to(PROJECT_ROOT))
        )

        if git_commit:
            mlflow.log_param("git_commit", git_commit)

        for split_name, split_df in mlp_metrics_df.groupby("split"):
            for row in split_df.itertuples(index=False):
                k = int(row.k)

                mlflow.log_metric(f"{split_name}_ndcg_at_{k}", float(row.ndcg))
                mlflow.log_metric(f"{split_name}_hit_rate_at_{k}", float(row.hit_rate))
                mlflow.log_metric(
                    f"{split_name}_recall_local_at_{k}", float(row.recall_local)
                )
                mlflow.log_metric(
                    f"{split_name}_precision_at_{k}", float(row.precision)
                )

                if split_name == "validation":
                    mlflow.log_metric(f"ndcg_at_{k}", float(row.ndcg))
                    mlflow.log_metric(f"hit_rate_at_{k}", float(row.hit_rate))
                    mlflow.log_metric(f"recall_local_at_{k}", float(row.recall_local))
                    mlflow.log_metric(f"precision_at_{k}", float(row.precision))

        mlflow.log_metric("best_validation_ndcg_at_10", float(best_validation_metric))
        mlflow.log_metric(
            "validation_ndcg_delta_vs_baseline_at_10",
            float(
                validation_comparison_df.loc[
                    validation_comparison_df["baseline"] == "mlp_temporal_v1",
                    "ndcg_delta_vs_baseline",
                ].iloc[0]
            ),
        )

        mlflow.log_dict(experiment_config, "config/experiment_config.json")

        for artifact_name, payload in mlflow_payloads.items():
            mlflow.log_dict(payload, f"config/{artifact_name}.json")

        mlflow.log_text(
            mlp_metrics_df.to_csv(index=False),
            "results/mlp_metrics.csv",
        )
        mlflow.log_text(
            validation_comparison_df.to_csv(index=False),
            "results/validation_comparison.csv",
        )
        mlflow.log_text(
            training_history_df.to_csv(index=False),
            "results/training_history.csv",
        )

    return tracking_uri

In [35]:
if RUN_MLFLOW:
    tracking_uri = log_mlp_run()
    print(f"Run registrada no MLflow: {tracking_uri}")
else:
    print("RUN_MLFLOW=False. Logging nao executado.")

🏃 View run mlp_temporal_v1_catfix at: https://dagshub.com/9MLET-GP16-Team/mlp-market-recommender-system.mlflow/#/experiments/5/runs/c80d4dd3eb2a41ae88f919ee8d7e2a51
🧪 View experiment at: https://dagshub.com/9MLET-GP16-Team/mlp-market-recommender-system.mlflow/#/experiments/5


Run registrada no MLflow: https://dagshub.com/9MLET-GP16-Team/mlp-market-recommender-system.mlflow


---

## 15. Decisao final e limitacoes

O MLP temporal foi escolhido como melhor ranker offline desta rodada porque superou o baseline principal `ordem_gerador_candidatos` em validacao e manteve desempenho proximo no teste.

Em validacao, o ganho em `ndcg@10` foi de `+0.032041`, saindo de `0.467768` para `0.499809`. O ganho tambem apareceu em `hit_rate@10`, `recall_local@10` e `precision@10`, indicando melhora consistente no top 10.

Embora `ordem_gerador_candidatos` seja um baseline forte, ele e uma heuristica diretamente acoplada a estrategia de candidate generation. Como a geracao prioriza recompra antes de similaridade, categoria e popularidade global, esse baseline tende a reproduzir a regra de negocio do gerador: colocar recompras no topo. Isso explica por que ele ficou muito proximo de `recompra_usuario` no notebook 05.

O MLP vale a pena porque aprende a reordenar os candidatos usando sinais adicionais: embeddings de usuario/produto/categoria e features temporais de recompra, recencia, frequencia e afinidade por categoria. Assim, ele nao depende apenas da ordem fixa criada pelo gerador de candidatos: quando os sinais de recencia e frequencia
indicam maior probabilidade de recompra, o modelo promove itens que a heuristica
deixaria mais abaixo no ranking.

**Limitacoes conhecidas:**

- O ganho sobre o melhor baseline e incremental (+0.032 de NDCG@10). O teto de
  recall (~70%) e imposto pela geracao de candidatos, nao pelo ranker — ampliar a
  cobertura das fontes de candidatos tende a valer mais que refinar o modelo.
- O protocolo avalia usuarios ja conhecidos (janelas temporais); nao ha medida de
  generalizacao para usuarios novos (cold start fica fora do escopo desta versao).
- A comparacao com/sem `candidate_rank`/`candidate_source` como feature segue como
  experimento futuro.
